# Three networks, one handwritten digit
Run the setup cells once. During the talk, rerun **Draw a digit** for the next example. Restart and run all cells to replay the same sequence.

In [ ]:
%matplotlib inline
from pathlib import Path
import os
if Path.cwd().name == 'notebooks':
    os.chdir('..')
assert Path('configs/experiment.json').is_file(), 'Open this notebook from the project folder.'
from IPython.display import Image, display
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import torch
from src.dataset import load_split
from src.utils import config, load_model, run_name

cfg = config()
x, y = load_split('t10k')
models = {rate: load_model(Path('checkpoints') / run_name(50000, 42, rate) / 'best.pt')
          for rate in cfg['rates']}
demo_seed = 42  # Set to None for a fresh random sequence.
rng = np.random.default_rng(demo_seed)


## Draw a digit
The bars show all ten softmax probabilities. Confidence is the probability assigned to the predicted digit.

In [ ]:
i = int(rng.integers(len(y)))
fig, axes = plt.subplots(1, 4, figsize=(13, 3))
axes[0].imshow(x[i, 0], cmap='gray')
axes[0].set_title(f'Test image {i} · true {int(y[i])}')
axes[0].axis('off')
with torch.inference_mode():
    for ax, (rate, model) in zip(axes[1:], models.items()):
        probabilities = model(x[i:i+1]).softmax(1)[0]
        digit = int(probabilities.argmax())
        ax.bar(range(10), probabilities.numpy(), color=['#087e80' if d == digit else '#a7bbc7' for d in range(10)])
        ax.set(xticks=range(10), ylim=(0, 1), xlabel='Digit', title=f'Drop {rate:g} · predicts {digit}\nConfidence {probabilities[digit]:.2%}')
fig.tight_layout()
plt.show()

## Return to the evidence
Compare the learning curves with the held-out error counts.

In [ ]:
for name in ['learning_loss', 'comparison', 'confusion']:
    display(Image(filename=f'results/figures/{name}.png', width=850))